출처: https://docs.langchain.com/oss/python/langchain/models#tool-calling

In [6]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

model = init_chat_model("gpt-5.2")

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

#### model.bind_tools()
- model.bind_tools()의 모델객체에 쿼리를 한다.
- 쿼리의 출력값 response에서 response.tool_calls는 아래를 자동으로 추출해준다.
1. 어떤 tool을 사용할 지
2. 그 tool에 어떤 입력값을 넣을 지

In [ ]:
# from rich import inspect
from rich.pretty import pprint

model_with_tools = model.bind_tools([get_weather])  

response = model_with_tools.invoke("What's the weather like in Boston?")
#----------------------------------------------------------------------------
# Q. response 안에 무엇이 있는가?
response_summary = {
    "content": response.content, # content는 tool_calls보다 우선적으로 나오지 않는다.
    "tool_calls": response.tool_calls,
    "finish_reason": response.response_metadata.get("finish_reason"),
    "model": response.response_metadata.get("model_name"),
    "usage": response.usage_metadata,
}
pprint(response_summary)
#----------------------------------------------------------------------------

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

{
│   'content': '',
│   'tool_calls': [
│   │   {
│   │   │   'name': 'get_weather',
│   │   │   'args': {'location': 'Boston'},
│   │   │   'id': 'call_iVfQcrYm8vfczwDCwrGb765w',
│   │   │   'type': 'tool_call'
│   │   }
│   ],
│   'finish_reason': 'tool_calls',
│   'model': 'gpt-4.1-2025-04-14',
│   'usage': {
│   │   'input_tokens': 63,
│   │   'output_tokens': 14,
│   │   'total_tokens': 77,
│   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   }
}

Tool: get_weather
Args: {'location': 'Boston'}


#### Tool execution loop
1. 모델로부터 tool_calls를 통해 tool과 args뽑기
2. tool에 args 넣어서 tool의 결과값 구하기
3. tool의 결과값을 다시 모델에게 줘서 최종 정답 뽑기

In [ ]:
from rich.pretty import pprint

# Bind (potentially multiple) tools to the model
model_with_tools = model.bind_tools([get_weather])

# Step 1: tool_call을 포함한 전체 ai_msg를 messages에 담기
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg) # TODO: 전부 다 넣는 이유가 궁금하다.
pprint(ai_msg)

# Step 2: tool 실행하고 messages에 담기
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result) 
    pprint(tool_result) # tool에 args를 넣은 결과

# Step 3: messages를 model에 넣어 최종 답변 넣기
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

AIMessage(
│   content='',
│   additional_kwargs={'refusal': None},
│   response_metadata={
│   │   'token_usage': {
│   │   │   'completion_tokens': 17,
│   │   │   'prompt_tokens': 131,
│   │   │   'total_tokens': 148,
│   │   │   'completion_tokens_details': {
│   │   │   │   'accepted_prediction_tokens': 0,
│   │   │   │   'audio_tokens': 0,
│   │   │   │   'reasoning_tokens': 0,
│   │   │   │   'rejected_prediction_tokens': 0
│   │   │   },
│   │   │   'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
│   │   },
│   │   'model_provider': 'openai',
│   │   'model_name': 'gpt-5.2-2025-12-11',
│   │   'system_fingerprint': None,
│   │   'id': 'chatcmpl-DGPEIXsKzpoLAN2rADqFojkdhZ1e2',
│   │   'service_tier': 'default',
│   │   'finish_reason': 'tool_calls',
│   │   'logprobs': None
│   },
│   id='lc_run--04a45f01-1acb-4f1a-9bf9-a1e364782a7e-0',
│   tool_calls=[
│   │   {
│   │   │   'name': 'get_weather',
│   │   │   'args': {'location': 'Boston'},
│   │   │   'id': 'call_HxPxo4cZVDwU2T0retKB5ovk',
│   │   │   'type': 'tool_call'
│   │   }
│   ],
│   usage_metadata={
│   │   'input_tokens': 131,
│   │   'output_tokens': 17,
│   │   'total_tokens': 148,
│   │   'input_token_details': {'audio': 0, 'cache_read': 0},
│   │   'output_token_details': {'audio': 0, 'reasoning': 0}
│   }
)

ToolMessage(content="It's sunny in Boston.", name='get_weather', tool_call_id='call_HxPxo4cZVDwU2T0retKB5ovk')

It’s sunny in Boston.


#### parallel tool calls

In [ ]:
from rich.pretty import pprint
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
    "What's the weather in Boston and Tokyo?"
)


# 여러개의 tool call은 한꺼번에 진행할 수 있음
pprint(response.tool_calls)
# [
#   {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_1'},
#   {'name': 'get_weather', 'args': {'location': 'Tokyo'}, 'id': 'call_2'},
# ]


# 아래 과정을 async로 진행하면 parallel하게 진행할 수 있음
results = []
for tool_call in response.tool_calls:
    if tool_call['name'] == 'get_weather':
        result = get_weather.invoke(tool_call)
    results.append(result)
    
pprint(results)

[
│   {
│   │   'name': 'get_weather',
│   │   'args': {'location': 'Boston, MA'},
│   │   'id': 'call_bG3hkciFmqd2vuBdbbv1yqSJ',
│   │   'type': 'tool_call'
│   },
│   {
│   │   'name': 'get_weather',
│   │   'args': {'location': 'Tokyo, Japan'},
│   │   'id': 'call_fCrVh13r7TzEDp4nvWk3R6ud',
│   │   'type': 'tool_call'
│   }
]

[
│   ToolMessage(
│   │   content="It's sunny in Boston, MA.",
│   │   name='get_weather',
│   │   tool_call_id='call_bG3hkciFmqd2vuBdbbv1yqSJ'
│   ),
│   ToolMessage(
│   │   content="It's sunny in Tokyo, Japan.",
│   │   name='get_weather',
│   │   tool_call_id='call_fCrVh13r7TzEDp4nvWk3R6ud'
│   )
]